# Final held-out comparison

Three frozen configurations, evaluated once each: baseline, Run 1, and quota-amended Run 2. This notebook reads saved results and recomputes their metrics from saved predictions. It makes no model/API calls and does not select or tune a pipeline.

In [1]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
sys.path.insert(0,str(ROOT/'src'))
summary_path=sorted((ROOT/'results/final_test').glob('*/summary.json'))[-1]
records=json.loads(summary_path.read_text(encoding='utf-8'))
table=json.loads((ROOT/'evaluation/FINAL_COMPARISON.json').read_text(encoding='utf-8'))
pd.DataFrame(table)

,model,split,n_answerable,n_negative,precision,recall,benchmark_precision,benchmark_recall,correct,false,correct_rate,false_rate
0,Run 1 (Gemini 3.5),train,140,23,0.656978,0.613776,0.656895,0.627910,17,17,0.739130,0.121429
1,Run 1 (Gemini 3.5),test,54,7,0.581446,0.569376,0.582111,0.586139,5,4,0.714286,0.074074
2,Run 2 (3.5 first stage + 3.1 recovery),train,140,23,0.730166,0.655898,0.730353,0.671938,0,0,0.000000,0.000000
3,Run 2 (3.5 first stage + 3.1 recovery),test,54,7,0.658684,0.629901,0.658614,0.637808,2,0,0.285714,0.000000
4,Baseline (local BM25),train,140,23,0.017433,0.181647,0.017594,0.192427,0,0,0.000000,0.000000
5,Baseline (local BM25),test,54,7,0.008050,0.138687,0.007581,0.138687,0,0,0.000000,0.000000


## All k values and evaluation counts

A cache is used only to reproduce already-recorded predictions. Evaluation attempts remain one per pipeline in the ledger. Original all-3.5 Run 2 is a separate historical train experiment; the held-out Run 2 has Gemini 3.1 recovery.

In [2]:
rows=[]
for record in records:
    for metric in record['test_metrics']['retrieval']:
        rows.append(dict(model=record['model'],**metric))
pd.DataFrame(rows)

,model,k,n_queries,precision,recall,hit,benchmark_precision,benchmark_recall
0,broad10,1,54,0.535623,0.451552,0.592593,0.535623,0.451552
1,broad10,3,54,0.581050,0.545337,0.722222,0.582305,0.562101
2,broad10,5,54,0.581446,0.569376,0.740741,0.582111,0.586139
3,broad10,10,54,0.579953,0.587894,0.759259,0.580456,0.611591
4,recovery20,1,54,0.616350,0.513115,0.666667,0.616350,0.513115
5,recovery20,3,54,0.657717,0.605863,0.777778,0.657575,0.613770
6,recovery20,5,54,0.658684,0.629901,0.796296,0.658614,0.637808
7,recovery20,10,54,0.657469,0.651476,0.833333,0.657638,0.666316
8,baseline,1,54,0.003074,0.018519,0.018519,0.003074,0.018519
9,baseline,3,54,0.007960,0.080719,0.111111,0.007626,0.080719


In [3]:
ledger=[json.loads(line) for line in (ROOT/'evaluation/test_runs.jsonl').read_text().splitlines()]
pd.DataFrame(ledger)

,run_id,model,status,at_utc,result_file
0,20261101T123300021745Z,broad10,started,2026-11-01T12:33:00.021745+00:00,NaN
1,20261101T123300021745Z,broad10,complete,2026-11-01T12:33:00.021745+00:00,results/final_test/20261101T123300021745Z/broa...
2,20261101T123300021745Z,recovery20,started,2026-11-01T12:33:02.338303+00:00,NaN
3,20261101T123300021745Z,recovery20,complete,2026-11-01T12:33:02.338303+00:00,results/final_test/20261101T123300021745Z/reco...
4,20261101T123300021745Z,baseline,started,2026-11-01T12:33:06.327837+00:00,NaN
5,20261101T123300021745Z,baseline,complete,2026-11-01T12:33:06.327837+00:00,results/final_test/20261101T123300021745Z/base...


## Reproduce metrics from saved predictions

The following cell calls only the shared scorer with a recorded-output lookup. It does not invoke retrieval, Gemini or model inference. This is a calculation audit, not new held-out tuning or a fresh pipeline test.

In [4]:
from heldout_evaluation import evaluate_test
from retrieval import load_corpus
documents,_=load_corpus(ROOT)
answerable=json.loads((ROOT/'data/prepared/test.json').read_text(encoding='utf-8'))
negatives=[r for r in json.loads((ROOT/'evaluation/unanswerable.json').read_text(encoding='utf-8')) if r['split']=='test']
class RecordedOutputs:
    def __init__(self,predictions):
        self.hits={r['query']:r['hits'] for group in ['answerable','negative'] for r in predictions[group]['predictions']}
    def search(self,query,k=5):
        return self.hits[query][:k]
for record in records:
    saved=json.loads((summary_path.parent/f"{record['model']}.json").read_text(encoding='utf-8'))
    audit=evaluate_test(RecordedOutputs(saved['predictions']),answerable,negatives,documents)
    for expected,actual in zip(record['test_metrics']['retrieval'],audit['answerable']['metrics']):
        for key in ['precision','recall','benchmark_precision','benchmark_recall','hit']:
            assert abs(expected[key]-actual[key])<1e-12
    assert abs(record['test_metrics']['false_abstention']-audit['answerable']['answerable_false_abstention_rate'])<1e-12
    assert abs(record['test_metrics']['correct_abstention']-audit['negative']['correct_abstention_rate'])<1e-12
    print(record['model'],'saved-prediction metric audit passed')

broad10 saved-prediction metric audit passed
recovery20 saved-prediction metric audit passed
baseline saved-prediction metric audit passed


## Interpretation

Run 2 has higher overlap scores and no false answerable abstentions on this split, but rejects fewer negatives. Run 1 is the more conservative recommendation for unsupported questions, while still missing the 5% false-abstention target. Seven negatives are a small sample. No production readiness is claimed.